# 第 15 章 · 从 mini 到你自己的 Harness（综合实战）

**这一章你会得到什么**：把前 14 章的东西合起来，**从零写一个麻雀虽小五脏俱全的 harness**——控制循环 + 完成检测 + 步数上限 + trajectory。写完你就能对着面试官讲清楚“harness 到底由什么组成”。

## 📖 对照源码（在 IDE 里打开这些文件，边看边跑）

回看这几处，确认你的最小 harness 和真实实现一一对应：

- `src/minisweagent/agents/default.py` **L88–122** — `run()`（对照你的 while 循环）
- `src/minisweagent/environments/local.py` **L45–56** — `_check_finished()`（对照你的 SUBMIT 检测）
- `src/minisweagent/agents/default.py` **L128–145** — `query()` 的 limit 检查（对照你的 step_limit）

> 快捷：代码格里 `函数名??` 直接打印源码；或用 `show_source("相对路径", 起始行, 结束行)`。

In [ ]:
import os, sys
from pathlib import Path
os.environ["MSWEA_SILENT_STARTUP"] = "1"
REPO = Path(r"/Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent")
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
import minisweagent
print("mini-SWE-agent:", minisweagent.__version__)

## 你要造的东西

一个 `MiniHarness`，具备真实 harness 的最小要素：
1. **循环**（run 重复 step）
2. **完成检测**（环境识别 submit 信号）
3. **步数上限**（超限则停）
4. **trajectory**（记录每一步，最后可导出）

下面给出脚手架，`TODO` 的地方你来补。参考实现在再下一格。

In [ ]:
class MiniHarness:
    def __init__(self, script, step_limit=10):
        self.script = script          # 假模型的固定回复序列
        self.step_limit = step_limit
        self.i = -1
        self.messages = []
        self.exit_status = None

    def query(self):
        self.i += 1
        reply = self.script[self.i]
        self.messages.append(reply)
        return reply

    def execute(self, action):
        # 完成检测：命令里出现 SUBMIT 就算提交
        if "SUBMIT" in action["command"]:
            self.exit_status = "Submitted"
            return None
        return f"[执行 {action['command']} 的输出]"

    def run(self, task):
        self.messages.append({"role": "user", "content": task})
        while True:
            # TODO 1: 超过 step_limit 就 self.exit_status = "LimitsExceeded" 并 break
            # TODO 2: reply = self.query()
            # TODO 3: result = self.execute(reply["action"])
            # TODO 4: 如果 exit_status 被设置了就 break
            # TODO 5: 否则把 {"role":"tool","content":result} 追加进 messages
            ...
        return {"exit_status": self.exit_status, "steps": self.i + 1}

## 参考实现 + 运行

先自己补上面那格，再跑这格核对。

In [ ]:
class MiniHarnessRef:
    def __init__(self, script, step_limit=10):
        self.script, self.step_limit = script, step_limit
        self.i, self.messages, self.exit_status = -1, [], None
    def query(self):
        self.i += 1
        reply = self.script[self.i]
        self.messages.append(reply)
        return reply
    def execute(self, action):
        if "SUBMIT" in action["command"]:
            self.exit_status = "Submitted"
            return None
        return f"[执行 {action['command']} 的输出]"
    def run(self, task):
        self.messages.append({"role": "user", "content": task})
        while True:
            if self.i + 1 >= self.step_limit:
                self.exit_status = "LimitsExceeded"; break
            reply = self.query()
            result = self.execute(reply["action"])
            if self.exit_status:
                self.messages.append({"role": "exit", "content": "done"}); break
            self.messages.append({"role": "tool", "content": result})
        return {"exit_status": self.exit_status, "steps": self.i + 1}

script = [
    {"role": "assistant", "content": "看目录", "action": {"command": "ls"}},
    {"role": "assistant", "content": "跑测试", "action": {"command": "pytest"}},
    {"role": "assistant", "content": "提交", "action": {"command": "echo SUBMIT"}},
]
h = MiniHarnessRef(script)
print("结果:", h.run("修个 bug"))
print("轨迹:", [m["role"] for m in h.messages])

## 实验：验证步数上限真的生效

给一个**永不提交**的剧本 + 小 step_limit，确认 harness 会靠 `LimitsExceeded` 停下来，而不是无限循环 / 越界。

In [ ]:
never_submit = [{"role": "assistant", "content": "循环", "action": {"command": "echo loop"}}] * 100
h2 = MiniHarnessRef(never_submit, step_limit=3)
print(h2.run("永不结束的任务"))
print("实际步数:", h2.i + 1, "（应等于 step_limit）")

## 对照真实 mini-SWE-agent

你这个 60 行的玩具，和 `default.py` 是**同构**的：

- 你的 `run` while 循环         ≈ `DefaultAgent.run()`（第 96 行的 `while True`）
- 你的 `execute` 里的 SUBMIT 检测 ≈ `LocalEnvironment._check_finished()`（魔法字符串）
- 你的 `exit_status`             ≈ 那条 `role="exit"` 消息的 `extra.exit_status`
- 你的 step_limit 分支            ≈ `query()` 里的 `LimitsExceeded`

真实版多出来的，全是**生产级加固**：真实模型接入、工具调用解析、格式错误重试、成本/时间上限、进程组超时、trajectory 落盘、多环境/多模型工厂、评测编排。每一样你在前 14 章都单独见过了。

## 面试题串讲（对着你的 MiniHarness 讲）

1. **Agent loop 在哪？** —— `run()` 的 while，不是 `step()`。
2. **谁发完成信号、谁识别、谁退出？** —— 模型发命令，环境 `_check_finished` 识别并抛 `Submitted`，`run()` 捕获后 break。
3. **为什么 action 解析放 Model？** —— provider-specific，Agent 只消费统一 action。
4. **超时怎么处理？** —— 单命令超时是 observation；Agent 超时是 `TimeExceeded`。
5. **可观测性怎么保证？** —— 每步 finally 存 trajectory；解析失败也持久化原始 response。
6. **怎么换模型/环境不改循环？** —— Protocol + 工厂，依赖能力不依赖继承。

## 结业标准

你能不看源码，从空文件写出一个带“循环 + 完成检测 + 步数上限 + 轨迹”的最小 harness，
并说清它和 `mini-SWE-agent` 各组件的对应关系。到这一步，你已经具备 agent-harness 岗位要考察的核心心智模型了。